In [ ]:
%load_ext autoreload
%autoreload 2

import os
os.chdir("../")

from ease_recommender import *
from npmi_recommender import *

import pickle as p

def create_mat(row, col, bool_to_int=True):
    # bool_to_int won't count duplicates in the same row, creates a different weighting basically
    if bool_to_int:
        data = np.ones_like(row, dtype=bool)
        return csr_matrix((data, (row, col))).astype(np.int64)
    else:
        data = np.ones_like(row, dtype=np.int64)
        return csr_matrix((data, (row, col)))

def check_if_all_terms_in_str(q, terms):
    for term in terms:
        if term not in q:
            return False

    return True

def get_cat2idx(category_type, D):
    if category_type == "track":
        return D["track2idx"]
    elif category_type == "album":
        return D["album2idx"]
    elif category_type == "artist":
        return D["artist2idx"]
    else:
        raise NotImplementedError

def find_match_using_terms(terms, cat2idx):
    matches = []
    for name in cat2idx.keys():
        if check_if_all_terms_in_str(name, terms):
            matches.append(name)

    if len(matches) > 1:
        raise Exception("Multiple matches found, filter down to a single match", matches)

    return matches[0]

print("loading cache data...")
D = p.load(open("cached_data/spotify_preprocessed.p", "rb"))

print("building csr matrices...")

# TODO: finish implementing track and album level recommendations

# track_mat = create_mat(D["playlist_indices"], D["track_indices"])
# album_mat = create_mat(D["playlist_indices"], D["album_indices"])
artist_mat = create_mat(D["playlist_indices"], D["artist_indices"])

print("done")

cat2idx = get_cat2idx("artist", D)
idx2cat = {v:k for k, v in cat2idx.items()}

# use two items that you believe are similar to optimize the value of lambda_

a_name = find_match_using_terms(["sgeir", "7xUZ4069zcyBM4Bn10NQ1c"], cat2idx)
# a_name = find_match_using_terms(["7fNWySjsDn74LCawyJ27EQ"], cat2idx)

a = cat2idx[a_name]
print(f"{a_name=}")
print(f"Num Rows: {artist_mat[:, a].sum()}")

# a = cat2idx[find_match_using_terms(["Fleet Foxes"], cat2idx)]

# b = cat2idx[find_match_using_terms(["Fleet Foxes"], cat2idx)]
# b = cat2idx[find_match_using_terms(["Bon Iver", "4LEiUm1SRbFMgfqnQTwUbQ"], cat2idx)]
# b = cat2idx[find_match_using_terms(["SOHN"], cat2idx)]

# b_name = find_match_using_terms(["7fNWySjsDn74LCawyJ27EQ"], cat2idx)
b_name = find_match_using_terms(["Highas"], cat2idx)
b = cat2idx[b_name]
print(f"{b_name=}")
print(f"Num Rows: {artist_mat[:, b].sum()}")

c_name = find_match_using_terms(["Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)"], cat2idx)
c = cat2idx[c_name]
print(f"{c_name=}")
print(f"Num Rows: {artist_mat[:, c].sum()}")

d_name = find_match_using_terms(["Fleet Foxes"], cat2idx)
d = cat2idx[d_name]
print(f"{d_name=}")
print(f"Num Rows: {artist_mat[:, d].sum()}")

e_name = find_match_using_terms(["Bon Iver (spotify:artist:4LEiUm1SRbFMgfqnQTwUbQ)"], cat2idx)
e = cat2idx[e_name]
print(f"{e_name=}")
print(f"Num Rows: {artist_mat[:, e].sum()}")

loading cache data...
building csr matrices...


In [ ]:
mat = artist_mat
mat = csr_array(mat)

n_users, n_items = mat.shape

X = mat.T @ mat
# X = X / n_users

X.shape

In [ ]:
import numpy as np
from scipy import sparse

def sparse_laplace_sppmi(X, alpha=1.0, normalize=False, zero_diag=True, non_neg=False):
    # Ensure input is CSR for fast row operations
    if not sparse.isspmatrix_csr(X):
        X = X.tocsr()
        
    # 1. Get Geometry of the Data
    # V: Vocabulary size (rows/cols)
    rows, cols = X.shape
    
    # N_raw: Total real observations
    N_raw = X.sum()
    
    # 2. Calculate "Smoothed" Marginals (The Global Statistics)
    # We pretend we added alpha to every cell, but we compute the sums analytically.
    
    # Virtual Total N = Real N + (alpha * Total Possible Cells)
    N_smoothed = N_raw + (alpha * rows * cols)
    
    # Raw marginals (sum of rows/cols)
    row_sums_raw = np.array(X.sum(axis=1)).flatten()
    col_sums_raw = np.array(X.sum(axis=0)).flatten()
    
    # Smoothed marginals = Raw Sum + (alpha * row_length)
    # Each row has 'cols' number of cells, so we add alpha * cols to the row sum
    P_x = (row_sums_raw + (alpha * cols)) / N_smoothed
    P_y = (col_sums_raw + (alpha * rows)) / N_smoothed
    
    # 3. Operate ONLY on Non-Zero Data (The Sparse Trick)
    # We extract the indices of existing data points to calculate their new PMI
    # efficiently, skipping the billions of zeros.
    
    # Create a copy to store results
    sppmi = X.copy().astype(np.float32)
    
    # Get indices of non-zero elements
    row_indices, col_indices = X.tocoo().nonzero()
    
    # Get the raw counts
    raw_counts = np.array(X.data)
    
    # Smooth the counts: Count_new = Count_raw + alpha
    smoothed_counts = raw_counts + alpha
    
    # Calculate P(x,y) for these specific entries
    P_xy = smoothed_counts / N_smoothed
    
    # 4. Vectorized PMI Calculation
    # PMI = log( P(x,y) / (P(x) * P(y)) )
    # Note: P_x[row_indices] grabs the specific P(x) for every non-zero entry
    
    # Numerator is P_xy
    # Denominator is P(x) * P(y)
    denominator = P_x[row_indices] * P_y[col_indices]
    
    # Calculate PMI (using log2)
    pmi_values = np.log2(P_xy / denominator)
    
    if normalize:
        pmi_values = pmi_values / -np.log2(P_xy)
    
    # 7. Update the matrix data
    sppmi.data = pmi_values
    
    if non_neg:
        sppmi.data = np.where(sppmi.data > 0, sppmi.data, 0)
    
    if zero_diag:
        sppmi.setdiag(np.zeros(sppmi.shape[0]))
    
    # 8. Clean up (Remove explicit zeros to keep matrix sparse)
    sppmi.eliminate_zeros()
    
    return sppmi

In [ ]:
alpha = .1 * .85

# sppmi = sparse_laplace_sppmi(X, alpha=alpha, non_neg=True)
# sppmi = sparse_laplace_sppmi(X, alpha=alpha, non_neg=False)
# sppmi = sparse_laplace_sppmi(X, alpha=alpha, non_neg=True, normalize=True)

sppmi = sparse_laplace_sppmi(X, alpha=alpha, zero_diag=False)

metric = (np.argsort(-sppmi[:, a].toarray()).tolist().index(b) + np.argsort(-sppmi[:, b].toarray()).tolist().index(a))/2
metric

In [ ]:
def get_metric(i, j, sppmi):
    return np.argsort(-sppmi[i].toarray()).tolist().index(j)

metrics = [
    get_metric(a, b, sppmi),
    get_metric(b, a, sppmi),
    
#     get_metric(b, c, sppmi),
#     get_metric(c, b, sppmi),
]

np.mean(metrics)

In [6]:
# chance of visiting a from b given n walks -> this is just pagerank basically
# 

In [7]:
# chance of visit relative to the stationary distribution... this is different, could be the baseline

In [8]:
from tqdm.auto import tqdm

In [9]:
import numpy as np
import scipy.sparse as sp
import rustworkx as rx
import time

class GraphAnalyzer:
    def __init__(self, csr_matrix, threshold=0.0):
        """
        Initializes the graph from a sparse CSR matrix.
        
        Args:
            csr_matrix: Scipy CSR array of unnormalized similarity scores.
            threshold: Minimum score to keep an edge. Crucial for performance 
                       if the similarity matrix is dense.
        """
        self.n_nodes = csr_matrix.shape[0]
        
        # 1. Efficient Conversion: CSR -> Rustworkx
        # We convert to COO format first to extract (row, col, weight) tuples efficiently
        # without iterating in pure Python.
        coo = csr_matrix.tocoo()
        
        # Apply threshold filtering
        if threshold > 0:
            mask = coo.data > threshold
            rows = coo.row[mask]
            cols = coo.col[mask]
            weights = coo.data[mask]
        else:
            rows = coo.row
            cols = coo.col
            weights = coo.data

        # 2. Build the graph
        # We use PyDiGraph (Directed) because RWR relies on directionality.
        # If your similarities are symmetric, edges (u,v) and (v,u) will both be added.
        self.graph = rx.PyDiGraph()
        self.graph.add_nodes_from(range(self.n_nodes))
        
        # Zip into edge tuples: (source, target, weight)
        # We cast weights to float to ensure Rust handles them correctly
        edge_list = list(zip(rows, cols, weights.astype(float)))
        
        # Bulk add is significantly faster than adding one by one
        self.graph.add_edges_from(tqdm(edge_list))
        
        print(f"✅ Graph initialized with {self.graph.num_nodes()} nodes and {self.graph.num_edges()} edges.")

    def random_walk_with_restart(self, seed_node, restart_prob=0.15):
        """
        Performs Random Walk with Restarts (RWR) from a specific seed node.
        
        RWR is mathematically equivalent to Personalized PageRank where the 
        personalization vector is concentrated entirely on the seed node.
        """
        # Damping factor alpha = 1 - restart_probability
        alpha = 1.0 - restart_prob
        
        # Personalization: {node_index: score}
        # We focus 100% of the personalization on the seed node
        personalization = {seed_node: 1.0}
        
        # rustworkx.pagerank automatically handles weight normalization 
        # (converting similarity scores to transition probabilities)
        scores = rx.pagerank(
            self.graph, 
            alpha=alpha, 
            weight_fn=lambda edge: edge, # Function to extract weight from edge payload
            personalization=personalization,
            tol=1e-6
        )
        
        # Convert dictionary {node: score} to a sorted list or array
        # Returning top 10 for immediate utility
        sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        return sorted_scores

#     def betweenness_centrality(self, k=None, normalized=True):
#         """
#         Computes Betweenness Centrality.
        
#         Args:
#             k: (int) If provided, approximates centrality using k pivot nodes.
#                Much faster for large graphs.
#         """
#         # This function runs in parallel threads in the Rust backend
#         bc_scores = rx.betweenness_centrality(
#             self.graph,
#             normalized=normalized,
#             endpoints=False, 
# #             weight_fn=lambda edge: 1.0 / edge if edge > 0 else 0 # Distance = 1/Similarity
#         )
        
#         return bc_scores

#     def shortest_path_to_all(self, source_node):
#         """
#         Calculates Dijkstra's shortest path distances from source to all other nodes.
#         Useful for analyzing connectivity spread.
#         """
#         # Computes shortest path lengths in parallel
#         # Note: For similarity graphs, "distance" is usually inverse of similarity.
#         distances = rx.dijkstra_shortest_path_lengths(
#             self.graph,
#             source_node,
#             edge_cost_fn=lambda edge: 1.0 / edge if edge > 0 else float('inf')
#         )
#         return distances

In [10]:
counts = X.diagonal()

In [11]:
keeps = counts >= counts[np.argsort(-counts)[5000]]

In [21]:
keep_inds = np.where(keeps)[0]

In [12]:
# keeps = counts >= counts[c]

In [13]:
X2 = X[keeps][:, keeps]

In [14]:
X2.shape

(5005, 5005)

In [34]:
# X3 = X2 * csr_array(1/X2.sum(axis=1)[:, None])

In [122]:
# # X3 = X2 / n_users
# X3 = sppmi[keeps][:, keeps].copy()
# # X3.data[X3.data < 0] = 0
# X3.data = 2**X3.data

# X3 = X3 / X3.sum()

# # X3 = X3 * csr_array(1/X3.sum(axis=1)[:, None])

In [146]:
# x = sppmi[keep_inds][:, keep_inds].toarray()

# assert np.allclose(x, x.T)

In [187]:
# pmi = sparse_laplace_sppmi(X, alpha=0, zero_diag=False)

In [195]:
# X3 = X2 / n_users
# X3 = X2 / n_users
X3 = sppmi[keep_inds][:, keep_inds].copy()
# X3.data[X3.data < 0] = 0
X3.data = 2**X3.data
# X3.data = 1/X3.data

X3 = X3 / X3.sum()

# X3 = X3 * csr_array(1/X3.sum(axis=1)[:, None])

In [196]:
def get_effective_resistance(L, u, v):
    """
    Calculates R_eff(u, v) using an iterative solver (Conjugate Gradient).
    Solves Lx = (e_u - e_v)
    R_eff = x[u] - x[v]
    """
    n = L.shape[0]
    
    # 1. Construct the right-hand side vector (e_u - e_v)
    # We use a sparse vector to save memory
    b = np.zeros(n)
    b[u] = 1
    b[v] = -1
    
    # 2. Solve the linear system Lx = b
    # L is singular, but b is orthogonal to the null space (components sum to 0),
    # so Conjugate Gradient (cg) converges to a valid solution.
    # tol=1e-6 is standard precision for this application.
    x, exit_code = spla.cg(L, b, tol=1e-6)
    
    if exit_code != 0:
        print("Warning: Solver did not converge perfectly.")
        
    # 3. Calculate resistance
    # R_eff = (e_u - e_v)^T * x  =>  x[u] - x[v]
    r_eff = x[u] - x[v]
    return r_eff

In [197]:
# ONLY run this if you have time and RAM
import scipy.linalg

# Convert to dense for full decomposition
L_dense = X3.todense()

# Get eigenvalues/vectors (hermitian=True for symmetric)
evals, evecs = scipy.linalg.eigh(L_dense)

# Compute pseudoinverse explicitly: sum( (1/lambda) * v * v.T )
# Ignore the first eigenvalue (which is 0)
L_dagger = np.dot(evecs[:, 1:] * (1.0 / evals[1:]), evecs[:, 1:].T)

# Now you can query L_dagger directly for the formula:
# R(u, v) = L_dagger[u,u] + L_dagger[v,v] - 2*L_dagger[u,v]

In [198]:
def R(u, v):
    return L_dagger[u,u] + L_dagger[v,v] - 2*L_dagger[u,v]

In [199]:
tgt = keep_inds.tolist().index(a)
scores = -np.array([R(tgt, i) for i in tqdm(range(X3.shape[0]))])
# scores = X3[tgt, :].toarray()

  0%|          | 0/5005 [00:00<?, ?it/s]

In [200]:
top_k_matches = [idx2cat[idx] for idx in keep_inds[np.argsort(-scores)]]

top_k_matches

['Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)',
 'DJ Drez (spotify:artist:5j3iObqG7iT7utWpTTmC7F)',
 "The O'Neill Brothers Group (spotify:artist:0cylxW7HGdK9xMdubw2oYW)",
 'Brian Crain (spotify:artist:6roo9MjIl9htSsX2Snnipb)',
 'BYU Vocal Point (spotify:artist:5zhxxb24WP6q6rbLHAn2UQ)',
 'Steve Anderson (spotify:artist:2FbZlUT418wQoIZ3IFTKeJ)',
 'Margaret Whiting (spotify:artist:5ZGDxHhju6eE8ja4IyGe87)',
 'Misty Edwards (spotify:artist:1D0eJV6n9INYqsRVhFoLZI)',
 'Sovereign Grace Music (spotify:artist:6MYKRhEIKscR8qdGlvpk9y)',
 'Giacomo Puccini (spotify:artist:0OzxPXyowUEQ532c9AmHUR)',
 'Sarah McMillan (spotify:artist:1taeI8pm5kXswB7L4603Tz)',
 'Johann Pachelbel (spotify:artist:62TD7509VQIxUe4WpwO0s3)',
 'Paul Baloche (spotify:artist:1jH3GuQCPI87UrS0hcScHr)',
 'Jadon Lavik (spotify:artist:4LIG4IMVuzGJjAhMxXtll1)',
 'Charlie Hall (spotify:artist:0Przo8VxOHYfZv9FMZWsWV)',
 'Starfield (spotify:artist:6sGx32zrdgpSQybEVnywmT)',
 'Leeland Mooring (spotify:artist:2ylW0JAtYmQpymRkrvWnJb)',
 '

In [204]:
X2

<Compressed Sparse Column sparse array of dtype 'int64'
	with 19564429 stored elements and shape (5005, 5005)>

In [207]:
import rustworkx as rx
import numpy as np
import math

# --- 1. Data Setup (The "Data Warehouse") ---
labels = [idx2cat[i] for i in keep_inds.tolist()]

n = len(labels)

# A. The Co-occurrence Matrix (Local Interactions)
# This represents the specific links we want to analyze in our graph.
# Rows = Source, Cols = Target
co_occurrence_matrix = X2.copy().toarray() 
print("Converting Matrix to Information Cost Graph...")

# Avoid division by zero with epsilon
epsilon = 1e-9

global_freqs = X2.diagonal()

# Create a matrix of "Source Probabilities" (Denominators)
# We reshape global_freqs to (N, 1) to broadcast across columns
source_totals = global_freqs.reshape(-1, 1)

# P(Target | Source) = Count(Source, Target) / GlobalCount(Source)
conditional_probs = co_occurrence_matrix / (source_totals + epsilon)

# Cost = -log2(Probability)
# We mask zeros to avoid log(0) errors (Infinite cost)
with np.errstate(divide='ignore'):
    cost_matrix = -np.log2(conditional_probs)

# Clean up infinite costs (where prob was 0)
cost_matrix[np.isinf(cost_matrix)] = 0.0

# --- 3. Build Graph from Matrix ---

graph = rx.PyDiGraph() # Directed because P(A|B) != P(B|A)

# Add nodes
for label in labels:
    graph.add_node(label)

# Add edges by iterating non-zero elements of the Cost Matrix
# (More efficient than iterating N*N for sparse matrices)
sources, targets = np.nonzero(co_occurrence_matrix)

edge_list = []
for u, v in zip(sources, targets):
    weight = cost_matrix[u, v]
    edge_list.append((u, v, weight))

graph.add_edges_from(edge_list)

# --- 4. Run All-Pairs Shortest Path ---

Converting Matrix to Information Cost Graph...


TypeError: digraph_all_pairs_dijkstra_shortest_paths() got an unexpected keyword argument 'weight_fn'

In [ ]:
# Result is Dict[Source_Idx][Target_Idx] = Path_List
all_paths = rx.all_pairs_dijkstra_shortest_paths(
    graph, 
    edge_cost_fn=lambda x: x
)

# --- 5. Analysis & Display ---

def get_path_cost(path_indices):
    total = 0
    for i in range(len(path_indices)-1):
        u, v = path_indices[i], path_indices[i+1]
        total += graph.get_edge_data(u, v)
    return total

def print_matrix_analysis(source_name, target_name):
    try:
        u = labels.index(source_name)
        v = labels.index(target_name)
    except ValueError:
        return

    if u in all_paths and v in all_paths[u]:
        path = all_paths[u][v]
        path_names = [labels[n] for n in path]
        cost = get_path_cost(path)
        
        # Convert cost back to probability for intuition
        # Cost = -log2(P)  =>  P = 2^(-Cost)
        total_prob = 2**(-cost)
        
        print(f"'{source_name}' -> '{target_name}'")
        print(f"   Path: {' -> '.join(path_names)}")
        print(f"   Info Cost: {cost:.2f} bits")
        print(f"   Flow Prob: {total_prob:.6f}")
        print("-" * 40)
    else:
        print(f"'{source_name}' and '{target_name}' are disconnected.")


In [ ]:
print_matrix_analysis(idx2cat[a], idx2cat[d])

In [ ]:

print(f"\n--- MATRIX-BASED RESULTS ---\n")

# Compare the "Generic" vs "Specific" paths
print_matrix_analysis("User 1", "Drama")
print_matrix_analysis("User 1", "Looper")

# Check Item-Item Inference (Titanic vs Looper)
# Note: This will likely fail or be infinite because our matrix 
# didn't define a path from Titanic -> Looper (only User -> Titanic).
# If we made the matrix Symmetric (Undirected), this would work.
print_matrix_analysis("Titanic", "Looper")

In [202]:
import rustworkx as rx
import math
import numpy as np

# --- 1. Setup Graph ---
graph = rx.PyGraph() # Undirected graph is usually better for PMI

# Helper to add nodes with properties
def add_node(name, category):
    return graph.add_node({"name": name, "category": category})

# Create Nodes
user = add_node("User 1", "User")

# Path A: The "Blockbuster" Trap
movie_pop = add_node("Titanic", "Movie")
tag_common = add_node("Drama", "Tag")

# Path B: The "Hidden Gem" (Target)
movie_rare = add_node("Primer", "Movie")
tag_niche = add_node("Time Travel", "Tag")
movie_rec = add_node("Looper", "Movie") # The recommendation

# Add edges (interactions)
# Indices match the order of creation: 0=User, 1=Titanic, 2=Drama, 3=Primer, 4=TimeTravel, 5=Looper
edges = [
    (0, 1), (1, 2), # User -> Titanic -> Drama
    (0, 3), (3, 4), # User -> Primer -> Time Travel
    (4, 5)          # Time Travel -> Looper
]
graph.add_edges_from_no_data(edges)

# --- 2. Calculate PMI Weights ---

total_edges = len(graph.edges())

# We need the degree of every node to calculate P(x)
# In rustworkx, we can get degrees efficiently
degrees = {node_idx: graph.degree(node_idx) for node_idx in graph.node_indices()}

# To simulate the real world, let's FORCE the degrees to be what they would be 
# in a massive database, rather than this tiny toy graph.
hypothetical_degrees = {
    0: 50,     # User has rated 50 things
    1: 10000,  # Titanic is super popular
    2: 5000,   # Drama is a huge category
    3: 20,     # Primer is a cult classic (rare)
    4: 15,     # Time Travel is specific
    5: 30      # Looper is moderately specific
}
# Update N to reflect the hypothetical universe size
N_hypothetical = 100000 

def calculate_pmi_cost(source, target):
    # P(x) = Degree(x) / N
    p_source = hypothetical_degrees.get(source, 1) / N_hypothetical
    p_target = hypothetical_degrees.get(target, 1) / N_hypothetical
    
    # P(x,y) = 1 / N (Assuming binary 'exists or not' link)
    p_edge = 1.0 / N_hypothetical
    
    # PMI = log( P(x,y) / (P(x)*P(y)) )
    # We add a small epsilon 1e-9 to avoid math errors if probabilities are 0
    pmi = math.log(p_edge / (p_source * p_target + 1e-9))
    
    # INVERT PMI for Dijkstra
    # We want Max PMI -> Min Cost.
    # Since PMI can be negative, a safe cost function is:
    # Cost = -PMI (if we handle negative weights) or simplified:
    # Cost = log(Degree(source)) + log(Degree(target)) 
    # Let's use the explicit "Information Cost" derived from PMI:
    # This is effectively minimizing the probability of the nodes occurring independently
    cost = math.log(hypothetical_degrees[source]) + math.log(hypothetical_degrees[target])
    return cost

# --- 3. Run Pathfinding ---

# We define the weight function dynamically
def pmi_weight_fn(edge):
    # rustworkx passes the edge payload, but we need indices. 
    # Since we didn't store indices in edge payload, we look them up.
    # For this snippet, we will re-calculate strictly based on the nodes involved.
    # Note: In real rx.dijkstra, the callback receives the Edge OBJECT or payload.
    # We will pre-calculate weights and store them in the edge payload for simplicity.
    return edge

# Update edges with PMI costs
for u, v in graph.edge_list():
    cost = calculate_pmi_cost(u, v)
    graph.update_edge(u, v, cost) # Store cost directly in edge

# Run Dijkstra
# Find path from User (0) to Looper (5) vs Drama (2)
paths = rx.dijkstra_shortest_paths(
    graph, 
    source=0, 
    weight_fn=lambda x: x # The weight is already the float cost we stored
)

# --- 4. Output Analysis ---

def analyze_path(target_idx):
    if target_idx not in paths:
        print(f"No path to {graph.get_node_data(target_idx)['name']}")
        return

    path_nodes = paths[target_idx]
    names = [graph.get_node_data(n)['name'] for n in path_nodes]
    
    # Sum costs
    total_cost = 0
    for i in range(len(path_nodes)-1):
        u, v = path_nodes[i], path_nodes[i+1]
        total_cost += graph.get_edge_data(u, v)
        
    print(f"Path to [{names[-1]}]:")
    print(f"  Via: {' -> '.join(names)}")
    print(f"  Total Information Cost: {total_cost:.2f}")
    print("  (Lower is better)\n")

print(f"--- PMI-BASED EXPLORATION (N={N_hypothetical}) ---\n")
analyze_path(2) # Drama
analyze_path(5) # Looper

--- PMI-BASED EXPLORATION (N=100000) ---

Path to [Drama]:
  Via: User 1 -> Titanic -> Drama
  Total Information Cost: 30.85
  (Lower is better)

Path to [Looper]:
  Via: User 1 -> Primer -> Time Travel -> Looper
  Total Information Cost: 18.72
  (Lower is better)



In [ ]:
import rustworkx as rx
import math

# --- 1. Setup Graph ---
graph = rx.PyGraph() 

# Helper to add nodes with properties
def add_node(name, category):
    return graph.add_node({"name": name, "category": category})

# Create Nodes
user = add_node("User 1", "User")

# Path A: The "Blockbuster" Trap
movie_pop = add_node("Titanic", "Movie")
tag_common = add_node("Drama", "Tag")

# Path B: The "Hidden Gem" (Target)
movie_rare = add_node("Primer", "Movie")
tag_niche = add_node("Time Travel", "Tag")
movie_rec = add_node("Looper", "Movie") 

# Add edges (interactions)
# Indices: 0=User, 1=Titanic, 2=Drama, 3=Primer, 4=TimeTravel, 5=Looper
edges = [
    (0, 1), (1, 2), # User -> Titanic -> Drama
    (0, 3), (3, 4), # User -> Primer -> Time Travel
    (4, 5)          # Time Travel -> Looper
]
graph.add_edges_from_no_data(edges)

# --- 2. Calculate PMI Weights ---

# Hypothetical degrees (Simulating the "Universe")
hypothetical_degrees = {
    0: 50,     # User
    1: 10000,  # Titanic (Popular)
    2: 5000,   # Drama (Common)
    3: 20,     # Primer (Rare)
    4: 15,     # Time Travel (Specific)
    5: 30      # Looper (Specific)
}
N_hypothetical = 100000 

def calculate_pmi_cost(source, target):
    # Cost = log(Degree(u)) + log(Degree(v))
    # This penalizes generic paths.
    cost = math.log(hypothetical_degrees[source]) + math.log(hypothetical_degrees[target])
    return cost

# Update edges with PMI costs in the payload
for u, v in graph.edge_list():
    cost = calculate_pmi_cost(u, v)
    graph.update_edge(u, v, cost) 

# --- 3. Run All-Pairs Pathfinding ---

print("Calculating Information Distance between ALL entities...")

# SWITCH: Dijkstra (One-to-All) -> All Pairs Dijkstra (All-to-All)
# Returns a Dict: { source_index: { target_index: [List of Node Indices] } }
all_paths = rx.all_pairs_dijkstra_shortest_paths(
    graph, 
    weight_fn=lambda x: x # The payload is the float cost
)

# --- 4. Output Analysis ---

def get_path_cost(path_indices):
    """Helper to recalculate cost of a path list"""
    total_cost = 0
    for i in range(len(path_indices)-1):
        u, v = path_indices[i], path_indices[i+1]
        total_cost += graph.get_edge_data(u, v)
    return total_cost

def analyze_relationship(source_idx, target_idx):
    s_data = graph.get_node_data(source_idx)
    t_data = graph.get_node_data(target_idx)
    
    # Check if path exists in the calculated map
    if source_idx in all_paths and target_idx in all_paths[source_idx]:
        path_nodes = all_paths[source_idx][target_idx]
        names = [graph.get_node_data(n)['name'] for n in path_nodes]
        cost = get_path_cost(path_nodes)
        
        print(f"[{s_data['name']}] <--> [{t_data['name']}]")
        print(f"  Route: {' -> '.join(names)}")
        print(f"  Info Cost: {cost:.2f}")
    else:
        print(f"[{s_data['name']}] and [{t_data['name']}] are disconnected.")
    print("-" * 40)

print(f"\n--- RESULTS (N={N_hypothetical}) ---\n")

# A. The User Recommendation (User -> Items)
print(">>> SCENARIO A: Recommendations (User Perspective)")
analyze_relationship(0, 5) # User -> Looper (The Hidden Gem)
analyze_relationship(0, 2) # User -> Drama (The Generic Noise)

# B. Item-to-Item Similarity (Item -> Item)
# This is the power of All-Pairs: we can check Item similarity 
# without the User being involved in the query.
print(">>> SCENARIO B: Item Similarity (Contextual)")
# Does "Titanic" relate to "Primer"? (Should be far/expensive)
analyze_relationship(1, 3) 

# Does "Primer" relate to "Looper"? (Should be close/cheap)
analyze_relationship(3, 5)

In [74]:
tgt = cat2idx[find_match_using_terms(["The Beatles (spotify:artist:3WrFJ7ztbogyGnTHbHJFl2)"], cat2idx)]

for item_id_a in [a, d, e, tgt]:
    for item_id_b in [a, d, e, tgt]:
        print(idx2cat[item_id_a], idx2cat[item_id_b], 
              R(keep_inds.tolist().index(item_id_a), keep_inds.tolist().index(item_id_b)))

Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c) Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c) 0.0
Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c) Fleet Foxes (spotify:artist:4EVpmkEwrLYEg6jIsiPMIb) 162.42515075491355
Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c) Bon Iver (spotify:artist:4LEiUm1SRbFMgfqnQTwUbQ) 338.45934647584073
Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c) The Beatles (spotify:artist:3WrFJ7ztbogyGnTHbHJFl2) 358.7212384101383
Fleet Foxes (spotify:artist:4EVpmkEwrLYEg6jIsiPMIb) Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c) 162.42515075491355
Fleet Foxes (spotify:artist:4EVpmkEwrLYEg6jIsiPMIb) Fleet Foxes (spotify:artist:4EVpmkEwrLYEg6jIsiPMIb) 0.0
Fleet Foxes (spotify:artist:4EVpmkEwrLYEg6jIsiPMIb) Bon Iver (spotify:artist:4LEiUm1SRbFMgfqnQTwUbQ) 475.57004950499174
Fleet Foxes (spotify:artist:4EVpmkEwrLYEg6jIsiPMIb) The Beatles (spotify:artist:3WrFJ7ztbogyGnTHbHJFl2) 456.9877442730619
Bon Iver (spotify:artist:4LEiUm1SRbFMgfqnQTwUbQ) Ásgeir (spotify:artist:7xUZ4069zcyB

In [ ]:
tgt = cat2idx[find_match_using_terms(["The Beatles (spotify:artist:3WrFJ7ztbogyGnTHbHJFl2)"], cat2idx)]

for item_id_a in [a, d, e, tgt]:
    for item_id_b in [a, d, e, tgt]:
        print(idx2cat[item_id_a], idx2cat[item_id_b], 
              R(keep_inds.tolist().index(item_id_a), keep_inds.tolist().index(item_id_b)))

In [49]:
tgt = cat2idx[find_match_using_terms(["The Beatles (spotify:artist:3WrFJ7ztbogyGnTHbHJFl2)"], cat2idx)]

for item_id_a in [a, d, e, tgt]:
    for item_id_b in [a, d, e, tgt]:
        print(idx2cat[item_id_a], idx2cat[item_id_b], 
              R(keep_inds.tolist().index(item_id_a), keep_inds.tolist().index(item_id_b)))

Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c) Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c) 0.0
Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c) Fleet Foxes (spotify:artist:4EVpmkEwrLYEg6jIsiPMIb) 0.10693458047313767
Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c) Bon Iver (spotify:artist:4LEiUm1SRbFMgfqnQTwUbQ) 0.019879832343021292
Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c) The Beatles (spotify:artist:3WrFJ7ztbogyGnTHbHJFl2) 0.02949156318278227
Fleet Foxes (spotify:artist:4EVpmkEwrLYEg6jIsiPMIb) Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c) 0.10693458047313767
Fleet Foxes (spotify:artist:4EVpmkEwrLYEg6jIsiPMIb) Fleet Foxes (spotify:artist:4EVpmkEwrLYEg6jIsiPMIb) 0.0
Fleet Foxes (spotify:artist:4EVpmkEwrLYEg6jIsiPMIb) Bon Iver (spotify:artist:4LEiUm1SRbFMgfqnQTwUbQ) 0.023086120961678072
Fleet Foxes (spotify:artist:4EVpmkEwrLYEg6jIsiPMIb) The Beatles (spotify:artist:3WrFJ7ztbogyGnTHbHJFl2) 0.22330026561711547
Bon Iver (spotify:artist:4LEiUm1SRbFMgfqnQTwUbQ) Ásgeir (spotify:artist:7x

In [35]:
# np.allclose(X3.sum(axis=1), 1)

In [ ]:
def get_effective_resistance(L, u, v):
    """
    Calculates R_eff(u, v) using an iterative solver (Conjugate Gradient).
    Solves Lx = (e_u - e_v)
    R_eff = x[u] - x[v]
    """
    n = L.shape[0]
    
    # 1. Construct the right-hand side vector (e_u - e_v)
    # We use a sparse vector to save memory
    b = np.zeros(n)
    b[u] = 1
    b[v] = -1
    
    # 2. Solve the linear system Lx = b
    # L is singular, but b is orthogonal to the null space (components sum to 0),
    # so Conjugate Gradient (cg) converges to a valid solution.
    # tol=1e-6 is standard precision for this application.
    x, exit_code = spla.cg(L, b, tol=1e-6)
    
    if exit_code != 0:
        print("Warning: Solver did not converge perfectly.")
        
    # 3. Calculate resistance
    # R_eff = (e_u - e_v)^T * x  =>  x[u] - x[v]
    r_eff = x[u] - x[v]
    return r_eff

In [ ]:
analyzer = GraphAnalyzer(X3)

In [22]:
scores = analyzer.random_walk_with_restart(a)

In [24]:
scores

[(9060, 0.1517269827591068),
 (325, 0.003496167315266312),
 (831, 0.0033434832881776874),
 (217, 0.0033401805111089103),
 (188, 0.002997387317589518),
 (390, 0.002697795935456204),
 (393, 0.0025844116407441583),
 (409, 0.0025736349551291823),
 (195, 0.002282660475817202),
 (985, 0.0022560111889089103),
 (840, 0.0021913340522799357),
 (870, 0.0021851811192070895),
 (728, 0.0021580105109698885),
 (809, 0.002146803984355017),
 (981, 0.0020789425678205665),
 (580, 0.002059610505813775),
 (338, 0.002034390092778231),
 (28, 0.001970626201998198),
 (582, 0.0019187475195077668),
 (1027, 0.001880329124099122),
 (14, 0.0018105416798197282),
 (246, 0.0018056680282509856),
 (746, 0.0017733854331987435),
 (2, 0.0017617512407879028),
 (583, 0.0017079945346890042),
 (208, 0.0017079295258767928),
 (326, 0.0017043631256191713),
 (178, 0.0017026159953181033),
 (22, 0.0016748021373048912),
 (196, 0.0016721676756522418),
 (789, 0.00162473484312259),
 (787, 0.0016245283067675927),
 (935, 0.0016043219004566

In [25]:
top_k, _ = zip(*scores)

top_k = list(top_k)

top_k.index(b)

2645

In [10]:
import rustworkx as rx
import numpy as np
import scipy.linalg
import math

class GraphPhysics:
    def __init__(self, mode="degree"):
        """
        mode: 
          'degree' -> Resistance based on node popularity (Information Cost)
          'conditional' -> Resistance based on transition probability
        """
        self.mode = mode
        self.graph = rx.PyGraph() # Effective Resistance requires Undirected Graph
        
    def add_node(self, name, degree_proxy=1):
        """
        degree_proxy: Manually setting 'popularity' for simulation.
        In a real graph, this would be the actual degree.
        """
        return self.graph.add_node({"name": name, "k": degree_proxy})
        
    def add_edge(self, u, v):
        self.graph.add_edge(u, v, None)

    def _calculate_resistance(self, u, v):
        """
        Calculates the Resistor Value (R) for an edge.
        Higher R = Harder to traverse = 'Higher Cost'
        """
        # Retrieve node data (our simulated degrees)
        u_data = self.graph.get_node_data(u)
        v_data = self.graph.get_node_data(v)
        k_u = u_data["k"]
        k_v = v_data["k"]

        if self.mode == "degree":
            # PMI-style Cost: log(k_u) + log(k_v)
            # Popular nodes are "High Resistance" (Bottlenecks)
            # We add 1.0 base resistance so R is never 0
            resistance = math.log(k_u + 1) + math.log(k_v + 1) + 1.0
            return resistance

        elif self.mode == "conditional":
            # Conditional Probability Cost: -log(P)
            # We treat the edge as a bi-directional pipe.
            # We calculate cost both ways and average them for the resistor value.
            
            # P(v|u) = 1 / Degree(u) (Simplification for unweighted)
            p_v_given_u = 1.0 / k_u
            p_u_given_v = 1.0 / k_v
            
            cost_forward = -math.log2(p_v_given_u) if p_v_given_u > 0 else 100
            cost_backward = -math.log2(p_u_given_v) if p_u_given_v > 0 else 100
            
            # Average resistance
            resistance = (cost_forward + cost_backward) / 2.0
            
            # Ensure non-zero resistance
            return max(resistance, 0.1)
            
    def compute_effective_resistance(self, start_node, end_node):
        num_nodes = len(self.graph)
        
        # 1. Build Adjacency Matrix where Weight = Conductance (1/R)
        # We start with zeros
        adj_matrix = np.zeros((num_nodes, num_nodes))
        
        for u, v in self.graph.edge_list():
            R = self._calculate_resistance(u, v)
            conductance = 1.0 / R 
            
            # Fill symmetric matrix
            adj_matrix[u, v] = conductance
            adj_matrix[v, u] = conductance
            
        # 2. Build Laplacian (L = D - A)
        # Degree matrix here is the sum of CONDUCTANCES (not simple edge count)
        degree_matrix = np.diag(np.sum(adj_matrix, axis=1))
        laplacian = degree_matrix - adj_matrix
        
        # 3. Compute Pseudoinverse (L+)
        # We use pinv because the graph is not "grounded" (L is singular)
        L_pinv = scipy.linalg.pinv(laplacian)
        
        # 4. Calculate R_eff
        # Formula: R_eff(u,v) = L+uu + L+vv - 2*L+uv
        r_eff = (L_pinv[start_node, start_node] + 
                 L_pinv[end_node, end_node] - 
                 2 * L_pinv[start_node, end_node])
                 
        return r_eff, L_pinv

In [12]:
import numpy as np
import scipy.sparse as sp
import rustworkx as rx
import time

class GraphAnalyzer:
    def __init__(self, csr_matrix, threshold=0.0):
        """
        Initializes the graph from a sparse CSR matrix.
        
        Args:
            csr_matrix: Scipy CSR array of unnormalized similarity scores.
            threshold: Minimum score to keep an edge. Crucial for performance 
                       if the similarity matrix is dense.
        """
        self.n_nodes = csr_matrix.shape[0]
        
        # 1. Efficient Conversion: CSR -> Rustworkx
        # We convert to COO format first to extract (row, col, weight) tuples efficiently
        # without iterating in pure Python.
        coo = csr_matrix.tocoo()
        
        # Apply threshold filtering
        if threshold > 0:
            mask = coo.data > threshold
            rows = coo.row[mask]
            cols = coo.col[mask]
            weights = coo.data[mask]
        else:
            rows = coo.row
            cols = coo.col
            weights = coo.data

        # 2. Build the graph
        # We use PyDiGraph (Directed) because RWR relies on directionality.
        # If your similarities are symmetric, edges (u,v) and (v,u) will both be added.
        self.graph = rx.PyDiGraph()
        self.graph.add_nodes_from(range(self.n_nodes))
        
        # Zip into edge tuples: (source, target, weight)
        # We cast weights to float to ensure Rust handles them correctly
        edge_list = list(zip(rows, cols, weights.astype(float)))
        
        # Bulk add is significantly faster than adding one by one
        self.graph.add_edges_from(edge_list)
        
        print(f"✅ Graph initialized with {self.graph.num_nodes()} nodes and {self.graph.num_edges()} edges.")

    def random_walk_with_restart(self, seed_node, restart_prob=0.15):
        """
        Performs Random Walk with Restarts (RWR) from a specific seed node.
        
        RWR is mathematically equivalent to Personalized PageRank where the 
        personalization vector is concentrated entirely on the seed node.
        """
        # Damping factor alpha = 1 - restart_probability
        alpha = 1.0 - restart_prob
        
        # Personalization: {node_index: score}
        # We focus 100% of the personalization on the seed node
        personalization = {seed_node: 1.0}
        
        # rustworkx.pagerank automatically handles weight normalization 
        # (converting similarity scores to transition probabilities)
        scores = rx.pagerank(
            self.graph, 
            alpha=alpha, 
            weight_fn=lambda edge: edge, # Function to extract weight from edge payload
            personalization=personalization,
            tol=1e-6
        )
        
        # Convert dictionary {node: score} to a sorted list or array
        # Returning top 10 for immediate utility
        sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        return sorted_scores

    def betweenness_centrality(self, k=None, normalized=True):
        """
        Computes Betweenness Centrality.
        
        Args:
            k: (int) If provided, approximates centrality using k pivot nodes.
               Much faster for large graphs.
        """
        # This function runs in parallel threads in the Rust backend
        bc_scores = rx.betweenness_centrality(
            self.graph,
            normalized=normalized,
            endpoints=False, 
#             weight_fn=lambda edge: 1.0 / edge if edge > 0 else 0 # Distance = 1/Similarity
        )
        
        return bc_scores

    def shortest_path_to_all(self, source_node):
        """
        Calculates Dijkstra's shortest path distances from source to all other nodes.
        Useful for analyzing connectivity spread.
        """
        # Computes shortest path lengths in parallel
        # Note: For similarity graphs, "distance" is usually inverse of similarity.
        distances = rx.dijkstra_shortest_path_lengths(
            self.graph,
            source_node,
            edge_cost_fn=lambda edge: 1.0 / edge if edge > 0 else float('inf')
        )
        return distances

In [ ]:
analyzer = GraphAnalyzer(sppmi)

target_node = a
print(f"\n--- Running RWR from Node {target_node} ---")
start = time.time()
# rwr_ranks = analyzer.random_walk_with_restart(target_node, restart_prob=0.15)
# rwr_ranks = analyzer.random_walk_with_restart(target_node, restart_prob=.4)
# rwr_ranks = analyzer.random_walk_with_restart(target_node, restart_prob=.5)

# rwr_ranks = analyzer.random_walk_with_restart(target_node, restart_prob=0)
# rwr_ranks = analyzer.random_walk_with_restart(target_node, restart_prob=.15)

# rwr_ranks = analyzer.random_walk_with_restart(target_node, restart_prob=1)
# rwr_ranks = analyzer.random_walk_with_restart(target_node, restart_prob=.1)


rwr_ranks = analyzer.random_walk_with_restart(target_node, restart_prob=.15)
# rwr_ranks = analyzer.random_walk_with_restart(target_node, restart_prob=.8)

# rwr_ranks = analyzer.random_walk_with_restart(target_node, restart_prob=.15)
# rwr_ranks = analyzer.random_walk_with_restart(target_node, restart_prob=0)

print(f"Done in {time.time() - start:.4f}s")
# print("Top 5 most related nodes:", rwr_ranks[:5])

top_k, _ = zip(*rwr_ranks)

top_k = list(top_k)

top_k.index(b)

In [ ]:
sim = GraphPhysics(mode="degree")

src = a
tgt = b

In [ ]:
r_eff, _ = sim.compute_effective_resistance(src, tgt)

In [173]:
import numpy as np
import scipy.sparse as sp
import rustworkx as rx
import time

class GraphAnalyzer:
    def __init__(self, csr_matrix, threshold=0.0):
        """
        Initializes the graph from a sparse CSR matrix.
        
        Args:
            csr_matrix: Scipy CSR array of unnormalized similarity scores.
            threshold: Minimum score to keep an edge. Crucial for performance 
                       if the similarity matrix is dense.
        """
        self.n_nodes = csr_matrix.shape[0]
        
        # 1. Efficient Conversion: CSR -> Rustworkx
        # We convert to COO format first to extract (row, col, weight) tuples efficiently
        # without iterating in pure Python.
        coo = csr_matrix.tocoo()
        
        # Apply threshold filtering
        if threshold > 0:
            mask = coo.data > threshold
            rows = coo.row[mask]
            cols = coo.col[mask]
            weights = coo.data[mask]
        else:
            rows = coo.row
            cols = coo.col
            weights = coo.data

        # 2. Build the graph
        # We use PyDiGraph (Directed) because RWR relies on directionality.
        # If your similarities are symmetric, edges (u,v) and (v,u) will both be added.
        self.graph = rx.PyDiGraph()
        self.graph.add_nodes_from(range(self.n_nodes))
        
        # Zip into edge tuples: (source, target, weight)
        # We cast weights to float to ensure Rust handles them correctly
        edge_list = list(zip(rows, cols, weights.astype(float)))
        
        # Bulk add is significantly faster than adding one by one
        self.graph.add_edges_from(edge_list)
        
        print(f"✅ Graph initialized with {self.graph.num_nodes()} nodes and {self.graph.num_edges()} edges.")

    def random_walk_with_restart(self, seed_node, restart_prob=0.15):
        """
        Performs Random Walk with Restarts (RWR) from a specific seed node.
        
        RWR is mathematically equivalent to Personalized PageRank where the 
        personalization vector is concentrated entirely on the seed node.
        """
        # Damping factor alpha = 1 - restart_probability
        alpha = 1.0 - restart_prob
        
        # Personalization: {node_index: score}
        # We focus 100% of the personalization on the seed node
        personalization = {seed_node: 1.0}
        
        # rustworkx.pagerank automatically handles weight normalization 
        # (converting similarity scores to transition probabilities)
        scores = rx.pagerank(
            self.graph, 
            alpha=alpha, 
            weight_fn=lambda edge: edge, # Function to extract weight from edge payload
            personalization=personalization,
            tol=1e-6
        )
        
        # Convert dictionary {node: score} to a sorted list or array
        # Returning top 10 for immediate utility
        sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        return sorted_scores

    def betweenness_centrality(self, k=None, normalized=True):
        """
        Computes Betweenness Centrality.
        
        Args:
            k: (int) If provided, approximates centrality using k pivot nodes.
               Much faster for large graphs.
        """
        # This function runs in parallel threads in the Rust backend
        bc_scores = rx.betweenness_centrality(
            self.graph,
            normalized=normalized,
            endpoints=False, 
#             weight_fn=lambda edge: 1.0 / edge if edge > 0 else 0 # Distance = 1/Similarity
        )
        
        return bc_scores

    def shortest_path_to_all(self, source_node):
        """
        Calculates Dijkstra's shortest path distances from source to all other nodes.
        Useful for analyzing connectivity spread.
        """
        # Computes shortest path lengths in parallel
        # Note: For similarity graphs, "distance" is usually inverse of similarity.
        distances = rx.dijkstra_shortest_path_lengths(
            self.graph,
            source_node,
            edge_cost_fn=lambda edge: 1.0 / edge if edge > 0 else float('inf')
        )
        return distances

In [165]:
sppmi_coo = sppmi.tocoo()

# m = sppmi_coo.data >= 4.5
m = sppmi_coo.data >= 5

# sppmi_sparse = csr_array((sppmi_coo.data[m], (sppmi_coo.row[m], sppmi_coo.col[m])), shape=sppmi.shape)
sppmi_sparse = csr_array((2**sppmi_coo.data[m], (sppmi_coo.row[m], sppmi_coo.col[m])), shape=sppmi.shape)

normalizer = sppmi_sparse.sum(axis=1)[:, None]
normalizer[normalizer > 0] = 1/normalizer[normalizer > 0]

sppmi_sparse = sppmi_sparse * normalizer

sppmi_sparse = sppmi_sparse.tocsr()

sppmi_sparse.sum(axis=1)

array([1., 1., 0., ..., 0., 0., 0.], shape=(295860,))

In [166]:
sppmi_sparse.nnz / sppmi.nnz

0.021356694223511948

In [175]:
analyzer = GraphAnalyzer(sppmi_sparse)

target_node = a
print(f"\n--- Running RWR from Node {target_node} ---")
start = time.time()
# rwr_ranks = analyzer.random_walk_with_restart(target_node, restart_prob=0.15)
# rwr_ranks = analyzer.random_walk_with_restart(target_node, restart_prob=.4)
# rwr_ranks = analyzer.random_walk_with_restart(target_node, restart_prob=.5)

# rwr_ranks = analyzer.random_walk_with_restart(target_node, restart_prob=0)
# rwr_ranks = analyzer.random_walk_with_restart(target_node, restart_prob=.15)

# rwr_ranks = analyzer.random_walk_with_restart(target_node, restart_prob=1)
# rwr_ranks = analyzer.random_walk_with_restart(target_node, restart_prob=.1)


rwr_ranks = analyzer.random_walk_with_restart(target_node, restart_prob=.15)
# rwr_ranks = analyzer.random_walk_with_restart(target_node, restart_prob=.8)

# rwr_ranks = analyzer.random_walk_with_restart(target_node, restart_prob=.15)
# rwr_ranks = analyzer.random_walk_with_restart(target_node, restart_prob=0)

print(f"Done in {time.time() - start:.4f}s")
# print("Top 5 most related nodes:", rwr_ranks[:5])

top_k, _ = zip(*rwr_ranks)

top_k = list(top_k)

top_k.index(b)

✅ Graph initialized with 295860 nodes and 5219070 edges.


In [ ]:
bc = analyzer.betweenness_centrality(k=100)

In [76]:
scores = sppmi[a].toarray()
top_k = np.argsort(-scores).tolist()

top_k.index(b)

12

In [169]:
# top_k = top_k[:20]

In [170]:
top_k_matches = [idx2cat[idx] for idx in top_k]

top_k_matches

['Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)',
 'Highasakite (spotify:artist:5awQWdBpLqN2KFVRN8w56T)',
 'The Acid (spotify:artist:0bRtSoJSpQdnbB3dWrWprR)',
 'Dustin Tebbutt (spotify:artist:0z9hynUsIjf0ddI4uHqPWX)',
 'Big Scary (spotify:artist:4mLYW48jy9Pwv6KpT74Evf)',
 'Ed Tullett (spotify:artist:5VGsR5wapeJIuRPX26IeGn)',
 'Lo-Fang (spotify:artist:5EDkJDlRNcMs3ewliB24QA)',
 'S. Carey (spotify:artist:2LSJrlndCuTpdEluvYHc2E)',
 'Allman Brown (spotify:artist:239Y6QdFqVFfdsw6moqSEN)',
 'PHOX (spotify:artist:3ix4iw2URncSdE7X292bXy)',
 'Vök (spotify:artist:7oDTyDfeA2JE2jUZztkBj8)',
 'Volcano Choir (spotify:artist:6gAtOqhriLzOzb3Qqmg5kO)',
 'Novo Amor (spotify:artist:0rZp7G3gIH6WkyeXbrZnGi)',
 'Liza Anne (spotify:artist:426VSUSxx9puUYFgp7l7EQ)',
 'Snakadaktal (spotify:artist:0SdEkx5Ai2gl0W7pnhlsfy)',
 'The Careful Ones (spotify:artist:1DdAoWvETBUklcJCOISZx1)',
 'Matt Corby (spotify:artist:7CIW23FQUXPc1zebnO1TDG)',
 'Night Beds (spotify:artist:533wKOfkJylNSi6ntO1wXd)',
 'SOHN (spotify:arti

In [171]:
sppmi[a, b], sppmi[b, c]

(np.float64(5.15147708284573), np.float64(5.970563799185629))

In [30]:
(sppmi >= 5).sum()

np.int64(5219070)

In [29]:
sppmi.nnz / (sppmi >= 5).sum()

np.float64(46.82372606613822)

In [31]:
sppmi_coo = sppmi.tocoo()

In [32]:
sppmi_coo.row

array([     0,      0,      0, ..., 295859, 295859, 295859],
      shape=(244376304,), dtype=int32)

In [35]:
m = sppmi_coo.data >= 5
sppmi_sparse = csr_array((sppmi_coo.data[m], (sppmi_coo.row[m], sppmi_coo.col[m])), shape=sppmi.shape)

sppmi_sparse

In [42]:
top_k = 100

scores = sppmi[b].toarray()
scores[np.argsort(-scores)][:top_k]

array([10.40859166,  5.9705638 ,  5.92283519,  5.75857571,  5.75417824,
        5.61457528,  5.58217614,  5.36321438,  5.27224304,  5.21260147,
        5.15147708,  5.13813142,  5.12376079,  5.07039063,  5.05707438,
        5.04604413,  5.01342481,  5.01133322,  5.00189371,  4.98003152,
        4.93685378,  4.92941766,  4.92931417,  4.92275872,  4.89887195,
        4.89817005,  4.88836995,  4.88481719,  4.88274651,  4.87599625,
        4.86551547,  4.85763243,  4.82310904,  4.81807206,  4.80875945,
        4.79157969,  4.78271779,  4.78002297,  4.75489629,  4.75232499,
        4.73954468,  4.73421814,  4.72559807,  4.71709601,  4.71435745,
        4.70607899,  4.70255034,  4.69924759,  4.69512501,  4.69293094,
        4.68268953,  4.67197963,  4.67090346,  4.66093319,  4.65682033,
        4.65408017,  4.64520893,  4.64149286,  4.61258096,  4.61189032,
        4.61114063,  4.60847171,  4.60529802,  4.59793757,  4.58756201,
        4.58300921,  4.57978145,  4.5760959 ,  4.57569146,  4.57

In [5]:
import numpy as np
from scipy import sparse

def sparse_pmi(pxy, px, row_indices, col_indices):
    denominator = px[row_indices] * px[col_indices]
    
    pmi = pxy.copy()
    pmi_values = np.log2(pxy.data / denominator)
    pmi.data = pmi_values
    
    return pmi

In [13]:
def get_metric(i, j, sppmi):
    return np.argsort(-sppmi[i].toarray()).tolist().index(j)

In [6]:
pxy = X / n_users

In [7]:
px = X.diagonal() / n_users

In [8]:
X_coo = X.tocoo()

row_indices = X_coo.row
col_indices = X_coo.col

In [9]:
pmi = sparse_pmi(pxy, px, row_indices, col_indices)

In [23]:
def normalize_pmi(pxy, px, row_indices, col_indices, pmi, norm_type="npmi"):
    row_norm = -np.log2(px[row_indices])
    col_norm = -np.log2(px[col_indices])
    
    if norm_type == "npmi":
        norm = -np.log2(pxy.data)
    elif norm_type == "mean":
        norm = (row_norm + col_norm)/2
    elif norm_type == "sum":
        norm = row_norm + col_norm
    elif norm_type == "min":
        norm = np.where(row_norm < col_norm, row_norm, col_norm)
    elif norm_type == "max":
        norm = np.where(row_norm > col_norm, row_norm, col_norm)
    
    pmi = pmi.copy()
    pmi.data /= norm
    
    return pmi

In [24]:
pmi2 = normalize_pmi(pxy, px, row_indices, col_indices, pmi, "sum")

metrics = [
    get_metric(a, b, pmi2),
    get_metric(b, a, pmi2),
    
#     get_metric(b, c, pmi2),
#     get_metric(c, b, pmi2),
]

np.mean(metrics)

np.float64(1070.5)

In [49]:
pxy[a, b] / (px[a] * px[b])

np.float64(39.20742196497797)

In [52]:
np.log2(pxy[a, b] / (px[a] * px[b]))

np.float64(5.293054877251469)

In [59]:
np.log2(pxy[c, c] / (px[c] * px[c])), -np.log2(pxy[c, c]), -np.log2(px[c])

(np.float64(13.781821449819493),
 np.float64(13.781821449819493),
 np.float64(13.781821449819493))

In [60]:
np.log2(pxy[b, c] / (px[b] * px[c])) / -np.log2(px[c])

np.float64(0.6107501137038661)

In [61]:
np.log2(pxy[b, c] / (px[b] * px[c])) / -np.log2(px[b])

np.float64(0.748528064568391)

In [63]:
np.log2(pxy[b, c] / (px[b] * px[c])) / max(-np.log2(px[c]), -np.log2(px[b]))

np.float64(0.6107501137038661)

In [55]:
np.log2(pxy[b, c] / (px[b] * px[c])) / np.log2(pxy[c, c] / (px[c] * px[c]))

np.float64(0.6107501137038661)

In [54]:
np.log2(pxy[b, c] / (px[b] * px[c])) / np.log2(pxy[b, b] / (px[b] * px[b]))

np.float64(0.748528064568391)

In [51]:
# pxy[a, c] / (px[a] * px[c])

In [47]:
pxy[b, c] / (px[b] * px[c])

np.float64(341.8569670449884)

In [48]:
pxy[c, b] / (px[c] * px[b])

np.float64(341.8569670449884)